# Clase 19 · Notebook 01
# Del texto a los números: tokenización, Bag of Words y embeddings

**Introducción al Deep Learning — Módulo 4, Unidad 1: Redes recurrentes (Parte I)**

Las redes de neuronas no entienden texto, solo números. Antes de poder alimentar una red
recurrente con lenguaje, tenemos que **convertir el texto en números**. En este cuaderno
recorremos las tres formas fundamentales de representar palabras:

1. **Tokenizar**: partir el texto en palabras y asignar un número (id) a cada una.
2. **One-hot / Bag of Words (BOW)**: representación *dispersa* basada en conteo de palabras.
3. **Embeddings**: vectores *densos* que la red aprende y que capturan significado.

> Ejecuta las celdas en orden con **Shift + Enter**. Lee cada celda de texto antes de correr el código.

## 1. Tokenización: del texto a una secuencia de números

In [1]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
tf.random.set_seed(42); np.random.seed(42)

frases = [
    "el gato duerme en el sofa",
    "el perro corre en el parque",
    "un gato y un perro juegan",
]

# TextVectorization aprende un vocabulario y convierte cada frase en enteros
vectorizar = tf.keras.layers.TextVectorization(output_mode="int", output_sequence_length=6)
vectorizar.adapt(frases)

vocab = vectorizar.get_vocabulary()
print("Vocabulario aprendido:")
for i, palabra in enumerate(vocab):
    print(f"  {i:2d} -> {palabra!r}")

print("\nLa frase 'el gato duerme en el sofa' se convierte en:")
print(vectorizar(["el gato duerme en el sofa"]).numpy()[0])

I0000 00:00:1784198197.738727       5 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1784198198.706410       5 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Vocabulario aprendido:
   0 -> ''
   1 -> '[UNK]'
   2 -> np.str_('el')
   3 -> np.str_('un')
   4 -> np.str_('perro')
   5 -> np.str_('gato')
   6 -> np.str_('en')
   7 -> np.str_('y')
   8 -> np.str_('sofa')
   9 -> np.str_('parque')
  10 -> np.str_('juegan')
  11 -> np.str_('duerme')
  12 -> np.str_('corre')

La frase 'el gato duerme en el sofa' se convierte en:
[ 2  5 11  6  2  8]


Cada palabra recibe un **número (token)**. Las posiciones `0` y `1` se reservan para el
relleno (*padding*) y las palabras desconocidas (`[UNK]`). La frase se convierte así en una
**secuencia de enteros de longitud fija** — justo lo que una red recurrente espera recibir.

## 2. One-hot / Bag of Words: contar palabras

Con `output_mode="multi_hot"` obtenemos, para cada frase, un vector que marca **qué palabras
del vocabulario aparecen** (1) y cuáles no (0). Esa es la idea de **Bag of Words**: cuenta
presencia de palabras e **ignora por completo el orden**.

In [2]:
bow = tf.keras.layers.TextVectorization(output_mode="multi_hot")
bow.adapt(frases)
vocab_bow = bow.get_vocabulary()
print("Tamaño del vocabulario:", len(vocab_bow))

vec = bow(["el gato duerme en el sofa"]).numpy()[0]
print("\nVector BOW (1 = la palabra aparece):")
print(vec.astype(int))
print("Palabras presentes:", [vocab_bow[i] for i, v in enumerate(vec) if v == 1])

Tamaño del vocabulario: 12

Vector BOW (1 = la palabra aparece):
[0 1 0 0 1 1 0 1 0 0 1 0]
Palabras presentes: [np.str_('el'), np.str_('gato'), np.str_('en'), np.str_('sofa'), np.str_('duerme')]


Dos problemas del enfoque Bag of Words:

- El vector es **disperso**: casi todo son ceros, y crece con el tamaño del vocabulario
  (decenas de miles de dimensiones en un caso real).
- **No captura significado**: no refleja que *gato* y *perro* son más parecidos entre sí
  que *gato* y *sofá*. Todas las palabras son igual de distintas.
- **Pierde el orden**: "el perro muerde al hombre" y "el hombre muerde al perro" dan el
  mismo vector.

## 3. Embeddings: vectores densos con significado

Una capa `Embedding` asigna a cada palabra un **vector denso** de pocas dimensiones. Esos
vectores empiezan siendo aleatorios, pero **la red los ajusta durante el entrenamiento**
(como cualquier otro peso), de modo que las palabras con significado parecido acaban con
vectores cercanos.

In [3]:
vocab_size = len(vectorizar.get_vocabulary())
DIM = 4   # cada palabra será un vector de 4 números (en la práctica, 50-300)

embedding = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=DIM)

secuencia = vectorizar(["el gato duerme en el sofa"])
vectores = embedding(secuencia)
print("Forma:", vectores.shape, " -> (1 frase, 6 palabras, 4 dimensiones por palabra)")
print("\nVector (embedding) de la palabra 'gato':")
idx_gato = vectorizar(["gato"]).numpy()[0][0]
print(embedding(np.array([idx_gato])).numpy()[0])

Forma: (1, 6, 4)  -> (1 frase, 6 palabras, 4 dimensiones por palabra)

Vector (embedding) de la palabra 'gato':
[-0.04305873 -0.00291236 -0.01728796  0.03614289]


### ¿Y word2vec?

`word2vec` (y familia: GloVe, fastText) son embeddings **preentrenados** sobre enormes
cantidades de texto. La idea clave: *"una palabra se define por su compañía"*. Palabras que
aparecen en contextos parecidos acaban con vectores parecidos, y esos vectores capturan
relaciones sorprendentes, como la famosa analogía:

`rey - hombre + mujer ≈ reina`

En la práctica, o bien la red **aprende** sus propios embeddings (celda anterior), o bien
**reutiliza** embeddings preentrenados.

## Resumen

| Representación | Cómo es | Captura significado | Captura orden |
|---|---|---|---|
| **Tokenización** | secuencia de ids enteros | no | sí |
| **One-hot / BOW** | vector disperso de conteo | no | no |
| **Embedding** | vector denso aprendido | **sí** | (con la red recurrente) |

- **Tokenizar** convierte el texto en una secuencia de números.
- **BOW** cuenta palabras pero ignora orden y similitud.
- Los **embeddings** dan vectores densos que capturan el significado y son la entrada
  natural de una red recurrente.

En el **Notebook 02** veremos cómo una red recurrente procesa esas secuencias **manteniendo
memoria** de lo que ha visto antes.